<a href="https://colab.research.google.com/github/Andrii2004/SDXL_finetunning/blob/main/SDXL_Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download files

In [ ]:
!pip install bitsandbytes transformers accelerate wandb xformers peft -q

!pip install git+https://github.com/huggingface/diffusers.git -q

!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/requirements_sdxl.txt
!wget https://github.com/myredex/functions/raw/master/files_helpers_functions.py

!pip install -r requirements_sdxl.txt

import locale
locale.getpreferredencoding = lambda: "UTF-8"

from huggingface_hub import notebook_login
notebook_login()

!accelerate config default

In [ ]:
import os
import json
import shutil
from google.colab import drive

drive.mount('/content/drive')

images_src = '/content/drive/MyDrive/cossack_images'
labels_src = '/content/drive/MyDrive/cossack_images_labels'
dest_folder = '/content/train_data'
images_dest = os.path.join(dest_folder, 'images')
metadata_file = os.path.join(dest_folder, 'metadata.jsonl')

os.makedirs(images_dest, exist_ok=True)
print(f"Directory created (if missing): {images_dest}")

def copy_images(src, dest):
    if not os.path.exists(src):
        print(f"Error: Source directory not found: {src}")
        return
    files = os.listdir(src)
    for file_name in files:
        if file_name.lower().endswith('.png'):
            shutil.copy(os.path.join(src, file_name), dest)

copy_images(images_src, images_dest)

all_images = {f for f in os.listdir(images_dest) if f.lower().endswith('.png')}
all_labels_files = [f for f in os.listdir(labels_src) if f.lower().endswith('.txt')]

metadata_entries = []
images_basenames = {os.path.splitext(f)[0] for f in all_images}
labels_basenames = set()

for txt_file in all_labels_files:
    base = os.path.splitext(txt_file)[0]
    labels_basenames.add(base)
    txt_path = os.path.join(labels_src, txt_file)
    with open(txt_path, 'r', encoding='utf-8') as f:
        caption = f.read().strip()
    image_file = base + '.png'
    if image_file in all_images:
        metadata_entries.append({
            "file_name": "images/"+image_file,
            "caption": caption
        })

missing_labels = images_basenames - labels_basenames
missing_images = labels_basenames - images_basenames

if not missing_labels and not missing_images:
    print("Success: Every image has a corresponding caption.")
else:
    if missing_labels:
        print(f"Error: Images without captions ({len(missing_labels)}):")
        for name in sorted(missing_labels):
            print(f"   - {name}.png")
    if missing_images:
        print(f"Warning: Captions without images ({len(missing_images)}):")
        for name in sorted(missing_images):
            print(f"   - {name}.txt")

with open(metadata_file, 'w', encoding='utf-8') as f:
    for entry in metadata_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"\nTraining data prepared:")
print(f"- Images directory: {images_dest}")
print(f"- Metadata file: {metadata_file}")
print(f"- Total pairs: {len(metadata_entries)}")

# Train LoRA

In [ ]:
!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/content/train_data" \
  --caption_column="caption" \
  --resolution=1024 \
  --rank=16 \
  --train_batch_size=3 \
  --num_train_epochs=350 \
  --checkpointing_steps=200 \
  --learning_rate=5e-05 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=50 \
  --mixed_precision="bf16" \
  --gradient_accumulation_steps=3 \
  --gradient_checkpointing \
  --output_dir="sdxl-lora-cossacks_v5" \
  --push_to_hub \
  --validation_prompt="<cossack_cartoon_style>, bright flat colors, bold black outlines, 2D animation frame of three Zaporozhian Cossacks in a green field, blue sky, high quality" \
  --validation_epochs=20 \
  --seed="0" \
  --report_to="wandb"

# Generate image

In [ ]:
import os
import torch
import random
import numpy as np
from diffusers import DiffusionPipeline, AutoencoderKL, EulerDiscreteScheduler

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    vae=vae
)

pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

pipe.load_lora_weights("/content/sdxl-lora-cossacks_v5")
pipe = pipe.to("cuda")

generator = torch.Generator(device="cuda").manual_seed(SEED)

prompt = "<cossack_cartoon_style>, frame showing a row of large, blue cannon on a dark green field. Pointing toward a desolate, hazy landscape under a dark and ominous sky filled with heavy clouds. The entire scene is rendered with the characteristic bold black outlines and vibrant, flat colors of the animation style."
result = pipe(
    prompt=prompt,
    cross_attention_kwargs={"scale": 1},
    num_inference_steps=50,
    guidance_scale=7.5,
    height=1024,
    width=1024,
    generator=generator
)

image = result.images[0]
image